
# 🌸 Project 22: My Little Journal
## Notebook 03 — Baseline Model Training

**Objective:** Train and evaluate a baseline multi-label emotion classification model using TF-IDF and Logistic Regression.

**Dataset:** GoEmotions (Preprocessed)



### 1. Load Preprocessed Data

Load cleaned text datasets and multi-hot encoded emotion labels from Notebook 02.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Define project directories
project_dir = Path.cwd().parent
processed_dir = project_dir / "data" / "processed"

# Load cleaned datasets
train_df = pd.read_pickle(processed_dir / "train.pkl")
val_df = pd.read_pickle(processed_dir / "validation.pkl")
test_df = pd.read_pickle(processed_dir / "test.pkl")

# Load multi-hot encoded labels
y_train = np.load(processed_dir / "y_train.npy")
y_val = np.load(processed_dir / "y_val.npy")
y_test = np.load(processed_dir / "y_test.npy")

# Prepare text inputs
X_train = train_df["text"].astype(str)
X_val = val_df["text"].astype(str)
X_test = test_df["text"].astype(str)

# Verify dimensions
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


Training: (43284,) (43284, 28)
Validation: (5416,) (5416, 28)
Test: (5427,) (5427, 28)



### 2. TF-IDF Feature Extraction

Convert text into numerical features using TF-IDF with unigrams and bigrams. Fit the vectorizer only on training data to prevent data leakage.


In [2]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

# Learn vocabulary from training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Apply the same vocabulary to validation data
X_val_tfidf = tfidf.transform(X_val)

# Check feature dimensions
print("Training:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


Training: (43284, 20000)
Validation: (5416, 20000)
Vocabulary size: 20000



### 3. Baseline Model Training

Train a One-vs-Rest Logistic Regression classifier to predict multiple emotions from TF-IDF text features.


In [4]:

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import numpy as np
import time

# Create writable copies of training data
X_train_safe = X_train_tfidf.copy()
y_train_safe = np.array(y_train, copy=True)

# Initialize Logistic Regression
base_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

# Multi-label classification
model = OneVsRestClassifier(
    base_model,
    n_jobs=1
)

# Train model
start_time = time.time()

model.fit(X_train_safe, y_train_safe)

training_time = time.time() - start_time

print("Model training completed!")
print(f"Training time: {training_time:.2f} seconds")


Model training completed!
Training time: 4.53 seconds



### 4. Validation Predictions

Generate multi-label emotion predictions on the validation dataset to evaluate the baseline model.


In [5]:

# Predict emotions on validation data
y_val_pred = model.predict(X_val_tfidf)

# Check prediction dimensions
print("Prediction shape:", y_val_pred.shape)

# Compare one prediction with its actual labels
print("\nActual labels:", y_val[0])
print("Predicted labels:", y_val_pred[0])


Prediction shape: (5416, 28)

Actual labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
Predicted labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]



### 5. Baseline Model Evaluation

Evaluate multi-label classification performance using precision, recall, F1-score, and the proportion of predictions with no positive labels.


In [6]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Evaluate the baseline model
print("Micro Precision:", round(precision_score(
    y_val, y_val_pred, average="micro", zero_division=0
), 4))

print("Micro Recall:", round(recall_score(
    y_val, y_val_pred, average="micro", zero_division=0
), 4))

print("Micro F1:", round(f1_score(
    y_val, y_val_pred, average="micro", zero_division=0
), 4))

print("Macro F1:", round(f1_score(
    y_val, y_val_pred, average="macro", zero_division=0
), 4))

# Identify how many examples received no predicted emotions
empty_predictions = (y_val_pred.sum(axis=1) == 0).sum()

print("\nPredictions with no emotions:", empty_predictions)
print("Percentage:", round(
    empty_predictions / len(y_val_pred) * 100, 2
), "%")


Micro Precision: 0.7195
Micro Recall: 0.2735
Micro F1: 0.3963
Macro F1: 0.2153

Predictions with no emotions: 3074
Percentage: 56.76 %



### 6. Prediction Probability Analysis

Examine predicted emotion probabilities to understand the impact of the default classification threshold.


In [7]:

# Get prediction probabilities
y_val_proba = model.predict_proba(X_val_tfidf)

# Count examples with no probability >= 0.5
max_probabilities = y_val_proba.max(axis=1)

print("Mean highest probability:", round(max_probabilities.mean(), 4))
print("Median highest probability:", round(np.median(max_probabilities), 4))
print("Highest probability below 0.5:", (max_probabilities < 0.5).sum())

# Show first example's top 5 emotions
emotion_names = [
    "admiration", "amusement", "anger", "annoyance",
    "approval", "caring", "confusion", "curiosity",
    "desire", "disappointment", "disapproval", "disgust",
    "embarrassment", "excitement", "fear", "gratitude",
    "grief", "joy", "love", "nervousness", "optimism",
    "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

top_indices = np.argsort(y_val_proba[0])[::-1][:5]

for idx in top_indices:
    print(f"{emotion_names[idx]}: {y_val_proba[0, idx]:.4f}")


Mean highest probability: 0.4957
Median highest probability: 0.462
Highest probability below 0.5: 3074
neutral: 0.2171
curiosity: 0.1127
admiration: 0.0755
disappointment: 0.0602
annoyance: 0.0559



### 7. Classification Threshold Analysis

Evaluate how different probability thresholds affect precision, recall, F1-score, and empty predictions on the validation dataset.


In [8]:

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5]
results = []

for threshold in thresholds:
    predictions = (y_val_proba >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Micro Precision": precision_score(
            y_val, predictions, average="micro", zero_division=0
        ),
        "Micro Recall": recall_score(
            y_val, predictions, average="micro", zero_division=0
        ),
        "Micro F1": f1_score(
            y_val, predictions, average="micro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_val, predictions, average="macro", zero_division=0
        ),
        "Empty Predictions (%)": (
            predictions.sum(axis=1) == 0
        ).mean() * 100
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))


,Threshold,Micro Precision,Micro Recall,Micro F1,Macro F1,Empty Predictions (%)
0,0.2,0.5316,0.5479,0.5396,0.3732,3.9328
1,0.3,0.5994,0.4567,0.5184,0.3084,18.2607
2,0.4,0.6590,0.3667,0.4712,0.2599,37.3338
3,0.5,0.7195,0.2735,0.3963,0.2153,56.7578



### 7. Classification Threshold Analysis

Compare different probability thresholds to improve the balance between precision and recall on the validation dataset.


In [9]:

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5]
results = []

for threshold in thresholds:
    predictions = (y_val_proba >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Micro Precision": precision_score(
            y_val, predictions, average="micro", zero_division=0
        ),
        "Micro Recall": recall_score(
            y_val, predictions, average="micro", zero_division=0
        ),
        "Micro F1": f1_score(
            y_val, predictions, average="micro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_val, predictions, average="macro", zero_division=0
        ),
        "Empty Predictions (%)": (
            predictions.sum(axis=1) == 0
        ).mean() * 100
    })

threshold_results = pd.DataFrame(results)
display(threshold_results.round(4))


,Threshold,Micro Precision,Micro Recall,Micro F1,Macro F1,Empty Predictions (%)
0,0.2,0.5316,0.5479,0.5396,0.3732,3.9328
1,0.3,0.5994,0.4567,0.5184,0.3084,18.2607
2,0.4,0.6590,0.3667,0.4712,0.2599,37.3338
3,0.5,0.7195,0.2735,0.3963,0.2153,56.7578



**Threshold Analysis Findings:**

- The default threshold of 0.5 produced high precision but low recall.
- Lowering the threshold increased recall and reduced empty predictions.
- Among the tested thresholds, 0.2 achieved the highest Micro F1 (0.5396) and Macro F1 (0.3732).
- Threshold 0.2 is selected provisionally for the baseline model.
- Final performance must be evaluated on unseen test data.



### 8. Save Baseline Model

Save the trained Logistic Regression model, TF-IDF vectorizer, and selected classification threshold for future inference.


In [10]:

import joblib
import json

# Create model directory
model_dir = project_dir / "models" / "baseline"
model_dir.mkdir(parents=True, exist_ok=True)

# Save trained model and vectorizer
joblib.dump(model, model_dir / "logistic_regression.joblib")
joblib.dump(tfidf, model_dir / "tfidf_vectorizer.joblib")

# Save metadata
metadata = {
    "model": "TF-IDF + One-vs-Rest Logistic Regression",
    "threshold": 0.2,
    "emotion_names": emotion_names,
    "validation_micro_f1": 0.5396,
    "validation_macro_f1": 0.3732
}

with open(model_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Baseline model saved successfully!")
print("Location:", model_dir.resolve())


Baseline model saved successfully!
Location: C:\Projects\22_My_Little_Journal_app\models\baseline



### 9. Emotion Prediction on Custom Text

Test the trained baseline model on new sentences using the selected classification threshold of 0.2.


In [11]:

# Function to predict emotions from new text
def predict_emotions(text, threshold=0.2):
    # Convert text into TF-IDF features
    text_features = tfidf.transform([text])

    # Get predicted probabilities
    probabilities = model.predict_proba(text_features)[0]

    # Select emotions above the threshold
    predictions = [
        (emotion_names[i], round(float(prob), 4))
        for i, prob in enumerate(probabilities)
        if prob >= threshold
    ]

    # Sort by probability (highest first)
    predictions.sort(key=lambda x: x[1], reverse=True)

    return predictions


# Test with a new sentence
sample_text = "I am so happy and grateful today!"

print("Text:", sample_text)
print("Predicted emotions:", predict_emotions(sample_text))


Text: I am so happy and grateful today!
Predicted emotions: [('joy', 0.8232)]



### 10. Qualitative Prediction Analysis

Test the baseline model on different emotional expressions to examine its predictions and limitations.


In [12]:

test_sentences = [
    "I feel so happy and excited today!",
    "I am sad because I miss my friends.",
    "I'm really angry about what happened.",
    "I'm nervous about my exam tomorrow.",
    "I feel thankful for my family.",
    "Today was a normal day. Nothing special happened."
]

for text in test_sentences:
    print(f"\nText: {text}")
    print("Predictions:", predict_emotions(text))



Text: I feel so happy and excited today!
Predictions: [('joy', 0.8516), ('excitement', 0.7037)]

Text: I am sad because I miss my friends.
Predictions: [('sadness', 0.987), ('disappointment', 0.3014)]

Text: I'm really angry about what happened.
Predictions: [('curiosity', 0.5313)]

Text: I'm nervous about my exam tomorrow.
Predictions: [('neutral', 0.3268)]

Text: I feel thankful for my family.
Predictions: [('neutral', 0.3447)]

Text: Today was a normal day. Nothing special happened.
Predictions: [('neutral', 0.4242)]



**Qualitative Evaluation Findings:**

- The model correctly identified joy, excitement, and sadness in selected examples.
- It failed to recognize anger, nervousness, and gratitude in other examples.
- Some sentences were incorrectly classified as neutral.
- These errors may reflect class imbalance, limited contextual understanding, and domain differences.
- The baseline demonstrates useful emotion recognition capabilities but requires further improvement.



### 11. Per-Emotion Performance Analysis

Evaluate precision, recall, and F1-score for each emotion using the selected validation threshold.


In [13]:

from sklearn.metrics import classification_report

# Predict using our selected threshold
y_val_pred_02 = (y_val_proba >= 0.2).astype(int)

# Generate per-emotion classification report
report = classification_report(
    y_val,
    y_val_pred_02,
    target_names=emotion_names,
    output_dict=True,
    zero_division=0
)

# Convert report into a DataFrame
report_df = pd.DataFrame(report).T

# Display performance for the 28 emotions
emotion_report = report_df.loc[emotion_names]

display(
    emotion_report[
        ["precision", "recall", "f1-score", "support"]
    ].sort_values("f1-score", ascending=False).round(3)
)


,precision,recall,f1-score,support
gratitude,0.954,0.875,0.913,353.0
love,0.734,0.801,0.766,251.0
amusement,0.742,0.693,0.717,303.0
admiration,0.643,0.701,0.671,488.0
remorse,0.741,0.588,0.656,68.0
neutral,0.447,0.937,0.605,1765.0
optimism,0.716,0.483,0.577,209.0
joy,0.661,0.419,0.512,172.0
desire,0.718,0.364,0.483,77.0
curiosity,0.508,0.368,0.427,247.0



**Per-Emotion Evaluation Findings:**

- Gratitude achieved the highest F1-score (0.913), followed by love (0.766) and amusement (0.717).
- Grief and relief achieved F1-scores of 0, indicating no correct positive predictions at the selected threshold.
- Nervousness and disappointment showed particularly low recall.
- Neutral achieved high recall (0.937) but relatively low precision (0.447), indicating many false-positive predictions.
- Performance varies considerably across emotion classes, highlighting the limitations of the baseline model.
- Further improvements could include class weighting, per-label threshold tuning, and transformer-based models.



### 12. Final Test Evaluation

Evaluate the trained baseline model on the held-out test dataset using the selected classification threshold of 0.2.


In [14]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Transform test text using the fitted TF-IDF vectorizer
X_test_tfidf = tfidf.transform(X_test)

# Predict probabilities on unseen test data
y_test_proba = model.predict_proba(X_test_tfidf)

# Apply the selected validation threshold
final_threshold = 0.2
y_test_pred = (y_test_proba >= final_threshold).astype(int)

# Calculate final evaluation metrics
print("FINAL TEST RESULTS")
print("-" * 35)

print("Micro Precision:", round(
    precision_score(y_test, y_test_pred, average="micro", zero_division=0), 4
))

print("Micro Recall:", round(
    recall_score(y_test, y_test_pred, average="micro", zero_division=0), 4
))

print("Micro F1:", round(
    f1_score(y_test, y_test_pred, average="micro", zero_division=0), 4
))

print("Macro F1:", round(
    f1_score(y_test, y_test_pred, average="macro", zero_division=0), 4
))

print("Empty Predictions (%):", round(
    (y_test_pred.sum(axis=1) == 0).mean() * 100, 2
))


FINAL TEST RESULTS
-----------------------------------
Micro Precision: 0.5277
Micro Recall: 0.5521
Micro F1: 0.5396
Macro F1: 0.3599
Empty Predictions (%): 3.89



## Conclusion

A baseline multi-label emotion classification model was trained using TF-IDF and One-vs-Rest Logistic Regression.

### Key Findings
- Training time: 4.53 seconds.
- The default threshold of 0.5 resulted in low recall.
- A threshold of 0.2 achieved the best validation Micro F1 among the tested thresholds.
- Final Test Micro F1: 0.5396.
- Final Test Macro F1: 0.3599.
- Empty predictions: 3.89%.
- The model performed consistently across validation and test datasets.
- Performance varied considerably between emotion categories, especially rare emotions.

### Limitations
- Some emotions were frequently misclassified or missed.
- The model was trained on short Reddit comments rather than personal journal entries.
- Further testing is needed before using predictions in a real journaling application.

### Next Steps
- Integrate the saved model into a Streamlit demo.
- Allow users to enter text and view predicted emotions.
- Display prediction probabilities and communicate model limitations.
- Explore advanced NLP models in future versions.
